# Gold Layer: Feature Engineering
## 🎯 ML Feature Generation Pipeline

**Purpose:** Calculate technical indicators and features for machine learning

**Key Features:**
- Versioned features (reproducible via `features_version`)
- Flexible schema using MAP<STRING, DOUBLE>
- Technical indicators: SMA, EMA, RSI, MACD, Bollinger Bands
- Returns and volatility calculations
- Tracked in `gold.runs` for full reproducibility

**Outputs:**
- `gold.features_daily` - Daily features by ticker
- `gold.runs` - Run metadata and versioning

---

### Architecture: Silver → Gold
```
silver.market_ohlcv
        ↓
  [Calculate Features]
        ↓
gold.features_daily (versioned)
```

In [ ]:
# Create run ID
run_id = str(uuid.uuid4())
run_start_time = datetime.now(timezone.utc)

# Get Delta table version for data_ref
try:
    silver_version = spark.sql("DESCRIBE HISTORY silver.market_ohlcv LIMIT 1").first()["version"]
    data_ref = f"silver.market_ohlcv@v{silver_version}"
except:
    data_ref = datetime.now(timezone.utc).isoformat()

print(f"🔖 Run ID: {run_id}")
print(f"📊 Data reference: {data_ref}")

# Create run record (we'll update status at the end)
run_record = spark.createDataFrame([{
    "run_id": run_id,
    "run_type": RUN_TYPE,
    "run_timestamp": run_start_time,
    "user_name": spark.sql("SELECT current_user()").first()[0],
    "model_name": None,
    "model_version": None,
    "data_ref": data_ref,
    "features_version": FEATURES_VERSION,
    "code_version": None,  # Would get from git in production
    "params_json": None,
    "constraints_json": None,
    "status": "running",
    "metrics_json": None,
    "error_message": None,
    "execution_time_seconds": None,
    "databricks_job_id": None,
    "notebook_path": "/Workspace/Repos/stock-market/notebooks/gold/03_feature_engineering",
    "cluster_id": None
}])

# Write to gold.runs
try:
    run_record.write.format("delta").mode("append").saveAsTable("gold.runs")
    print("✓ Run record created in gold.runs")
except:
    print("⚠️  Creating gold.runs table...")
    run_record.write.format("delta").mode("overwrite").saveAsTable("gold.runs")
    print("✓ gold.runs table created")

In [ ]:
# Read from silver layer
ohlcv_df = spark.sql("""
    SELECT ticker, date, open, high, low, close, volume, adjusted_close
    FROM silver.market_ohlcv
    ORDER BY ticker, date
""")

print(f"✓ Loaded {ohlcv_df.count()} OHLCV records from silver.market_ohlcv")

# Function to calculate features for each ticker
def calculate_features_for_ticker(pdf: pd.DataFrame) -> pd.DataFrame:
    """Calculate technical indicators using pandas_ta."""
    
    # Sort by date
    pdf = pdf.sort_values('date')
    
    # Simple Moving Averages
    pdf['sma_5'] = ta.sma(pdf['close'], length=5)
    pdf['sma_10'] = ta.sma(pdf['close'], length=10)
    pdf['sma_20'] = ta.sma(pdf['close'], length=20)
    pdf['sma_50'] = ta.sma(pdf['close'], length=50)
    
    # Exponential Moving Averages
    pdf['ema_12'] = ta.ema(pdf['close'], length=12)
    pdf['ema_26'] = ta.ema(pdf['close'], length=26)
    
    # RSI (Relative Strength Index)
    pdf['rsi_14'] = ta.rsi(pdf['close'], length=14)
    
    # MACD
    macd = ta.macd(pdf['close'])
    if macd is not None:
        pdf['macd'] = macd['MACD_12_26_9']
        pdf['macd_signal'] = macd['MACDs_12_26_9']
        pdf['macd_hist'] = macd['MACDh_12_26_9']
    
    # Bollinger Bands
    bbands = ta.bbands(pdf['close'], length=20)
    if bbands is not None:
        pdf['bb_upper'] = bbands['BBU_20_2.0']
        pdf['bb_middle'] = bbands['BBM_20_2.0']
        pdf['bb_lower'] = bbands['BBL_20_2.0']
        pdf['bb_width'] = (bbands['BBU_20_2.0'] - bbands['BBL_20_2.0']) / bbands['BBM_20_2.0']
    
    # Returns
    pdf['return_1d'] = pdf['close'].pct_change(1)
    pdf['return_5d'] = pdf['close'].pct_change(5)
    pdf['return_20d'] = pdf['close'].pct_change(20)
    
    # Volatility (rolling std of returns)
    pdf['volatility_20d'] = pdf['return_1d'].rolling(20).std()
    
    # Volume features
    pdf['volume_sma_20'] = ta.sma(pdf['volume'], length=20)
    pdf['volume_ratio'] = pdf['volume'] / pdf['volume_sma_20']
    
    # Price position relative to high/low
    pdf['price_position'] = (pdf['close'] - pdf['low']) / (pdf['high'] - pdf['low'])
    
    return pdf

# Apply to all tickers using Pandas UDF
schema = ohlcv_df.schema
for col_name in ['sma_5', 'sma_10', 'sma_20', 'sma_50', 'ema_12', 'ema_26', 'rsi_14',
                  'macd', 'macd_signal', 'macd_hist', 'bb_upper', 'bb_middle', 'bb_lower',
                  'bb_width', 'return_1d', 'return_5d', 'return_20d', 'volatility_20d',
                  'volume_sma_20', 'volume_ratio', 'price_position']:
    schema = schema.add(StructField(col_name, DoubleType(), True))

@F.pandas_udf(schema, F.PandasUDFType.GROUPED_MAP)
def calculate_features_udf(pdf):
    return calculate_features_for_ticker(pdf)

features_df = ohlcv_df.groupby("ticker").apply(calculate_features_udf)

print("✓ Features calculated for all tickers")

In [ ]:
# Calculate execution time
run_end_time = datetime.now(timezone.utc)
execution_time = (run_end_time - run_start_time).total_seconds()

# Update run status
spark.sql(f"""
    UPDATE gold.runs
    SET status = 'completed',
        execution_time_seconds = {execution_time},
        metrics_json = '{{"total_features": {gold_features_df.count()}, "feature_columns": {len(feature_columns)}}}'
    WHERE run_id = '{run_id}'
""")

print("✓ Run status updated to 'completed'")

print("\n" + "="*60)
print("✅ FEATURE ENGINEERING COMPLETE")
print("="*60)
print(f"  Run ID: {run_id}")
print(f"  Features version: {FEATURES_VERSION}")
print(f"  Execution time: {execution_time:.1f}s")
print(f"  Total features: {gold_features_df.count()}")
print(f"  Feature dimensions: {len(feature_columns)}")

# Show summary
spark.sql("""
    SELECT 
        COUNT(*) as total_rows,
        COUNT(DISTINCT ticker) as unique_tickers,
        COUNT(DISTINCT date) as unique_dates,
        MIN(date) as earliest_date,
        MAX(date) as latest_date
    FROM gold.features_daily
""").show()

print("\n📌 Features ready for ML training and predictions!")
print("   Next: Build LSTM model or run portfolio optimization")

## 4️⃣ Update Run Status and Summary

In [ ]:
# Convert features to MAP format
feature_columns = [
    'sma_5', 'sma_10', 'sma_20', 'sma_50', 'ema_12', 'ema_26', 'rsi_14',
    'macd', 'macd_signal', 'macd_hist', 'bb_upper', 'bb_middle', 'bb_lower',
    'bb_width', 'return_1d', 'return_5d', 'return_20d', 'volatility_20d',
    'volume_sma_20', 'volume_ratio', 'price_position',
    'open', 'high', 'low', 'close', 'volume'  # Include raw OHLCV as features too
]

# Create MAP from columns
gold_features_df = features_df.select(
    F.col("ticker"),
    F.col("date"),
    F.lit(FEATURES_VERSION).alias("features_version"),
    F.create_map(
        *[F.lit(col_name), F.col(col_name) for col_name in feature_columns]
    ).alias("features"),
    F.current_timestamp().alias("created_timestamp"),
    F.lit(run_id).alias("run_id")
)

# Remove rows with all null features (beginning of time series)
gold_features_df = gold_features_df.filter(F.size(F.col("features")) > 0)

print(f"✓ Prepared {gold_features_df.count()} feature records")
gold_features_df.show(5)

# Write to gold.features_daily
print("\n💾 Writing to gold.features_daily...")

gold_features_df.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("date") \
    .option("overwriteSchema", "true") \
    .option("replaceWhere", f"features_version = '{FEATURES_VERSION}'") \
    .saveAsTable("gold.features_daily")

print("✓ Features written to gold.features_daily")

# Optimize
spark.sql("OPTIMIZE gold.features_daily ZORDER BY (ticker)")
print("✓ Table optimized")

## 3️⃣ Convert to MAP Format and Write to Gold

Transform features into flexible MAP<STRING, DOUBLE> format for extensibility.

## 2️⃣ Load OHLCV Data and Calculate Features

Load from Silver layer and calculate technical indicators using pandas_ta.

## 1️⃣ Create Run Record

Track this feature engineering run in `gold.runs` for reproducibility.

In [ ]:
import pandas as pd
import pandas_ta as ta
import uuid
from datetime import datetime, timezone
from typing import Dict

from pyspark.sql import SparkSession, DataFrame, Window
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder.getOrCreate()

# Configuration
FEATURES_VERSION = "v1"  # Increment when changing feature logic
RUN_TYPE = "feature_engineering"

print(f"✓ Feature Engineering Pipeline")
print(f"  Features version: {FEATURES_VERSION}")
print(f"  Timestamp: {datetime.now(timezone.utc).isoformat()}")

In [ ]:
# Install technical analysis library
%pip install pandas-ta --quiet

dbutils.library.restartPython()